# Redes Neuronales II: aprendizaje paso a paso

**Estudiante:** Karen Sofía Delgado Calderón  
**Actividad:** R1-A2-S4  
**Curso:** Deep Learning - Conceptos  
**Entorno:** Google Colab / Python  

## ¿Qué demuestra este notebook?

La práctica comienza con una neurona muy sencilla y avanza hasta una red multicapa capaz de reconocer los diez dígitos de MNIST. El objetivo no es únicamente obtener una exactitud alta, sino entender qué ocurre durante el aprendizaje.

Se desarrollan cinco experimentos:

1. Perceptrón para datos separables.
2. Neurona sigmoide para clasificación binaria real.
3. Red multicapa programada con NumPy y backpropagation manual.
4. Clasificación de MNIST con TensorFlow + Keras.
5. Réplica de la arquitectura densa con PyTorch.

> Importante: se utilizan capas densas; no se emplean redes convolucionales.

## 1. Preparación del entorno

In [ ]:
# En Google Colab estas bibliotecas ya suelen estar instaladas.
# Si se ejecuta en otro entorno, se puede habilitar la siguiente línea:
# !pip install numpy matplotlib scikit-learn tensorflow torch

import os
import random

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
import torch
from sklearn.datasets import load_breast_cancer, make_blobs, make_moons
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    confusion_matrix,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

# La semilla hace que el muestreo y la inicialización sean repetibles.
SEED = 29
CAPAS_OCULTAS = (160, 80)

np.random.seed(SEED)
random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
torch.manual_seed(SEED)

print("TensorFlow:", tf.__version__)
print("PyTorch:", torch.__version__)
print("Semilla de trabajo:", SEED)

## 2. Funciones auxiliares

La función sigmoide transforma cualquier número real en un valor entre 0 y 1. Por eso puede interpretarse como una probabilidad en problemas de dos clases.

La entropía cruzada mide qué tan lejos está la probabilidad calculada de la etiqueta correcta. Durante el entrenamiento se busca disminuir esta pérdida.

In [ ]:
def sigmoide(z):
    '''Convierte una entrada real en una probabilidad entre 0 y 1.'''
    z = np.clip(z, -50, 50)  # Evita desbordamientos numéricos.
    return 1.0 / (1.0 + np.exp(-z))


def entropia_cruzada_binaria(y_real, probabilidad):
    '''Calcula la pérdida promedio para un problema binario.'''
    epsilon = 1e-9
    probabilidad = np.clip(probabilidad, epsilon, 1 - epsilon)
    return -np.mean(
        y_real * np.log(probabilidad)
        + (1 - y_real) * np.log(1 - probabilidad)
    )

## 3. Experimento 1 — Perceptrón desde cero

El perceptrón recibe dos características, calcula una suma ponderada y decide entre las clases 0 y 1. Cuando se equivoca, corrige sus pesos.

Este primer experimento utiliza dos grupos claramente separados. Su propósito es verificar que la regla básica de actualización funciona antes de estudiar modelos más complejos.

In [ ]:
# Se crean dos grupos artificiales fáciles de separar mediante una línea.
X_perceptron, y_perceptron = make_blobs(
    n_samples=320,
    centers=[(-2.2, -1.7), (2.0, 2.2)],
    cluster_std=0.65,
    random_state=SEED,
)

# Se agrega una columna de unos para representar el término de sesgo.
X_con_sesgo = np.column_stack([np.ones(len(X_perceptron)), X_perceptron])
pesos_perceptron = np.zeros(X_con_sesgo.shape[1])
tasa_aprendizaje = 0.08
errores_por_epoca = []

for epoca in range(35):
    errores = 0

    for caracteristicas, etiqueta_real in zip(X_con_sesgo, y_perceptron):
        salida = caracteristicas @ pesos_perceptron
        prediccion = int(salida >= 0)

        # Solo se modifican los pesos cuando la predicción es incorrecta.
        correccion = tasa_aprendizaje * (etiqueta_real - prediccion)
        pesos_perceptron += correccion * caracteristicas
        errores += int(correccion != 0)

    errores_por_epoca.append(errores)
    if errores == 0:
        break

pred_perceptron = (X_con_sesgo @ pesos_perceptron >= 0).astype(int)
acc_perceptron = accuracy_score(y_perceptron, pred_perceptron)

print(f"Épocas ejecutadas: {len(errores_por_epoca)}")
print(f"Exactitud del perceptrón: {acc_perceptron:.2%}")
print("Pesos aprendidos:", np.round(pesos_perceptron, 4))

# Visualización de los datos y de la frontera aprendida.
plt.figure(figsize=(7, 5))
plt.scatter(
    X_perceptron[:, 0], X_perceptron[:, 1],
    c=y_perceptron, cmap="coolwarm", edgecolor="white", s=45
)

x_linea = np.linspace(X_perceptron[:, 0].min(), X_perceptron[:, 0].max(), 100)
y_linea = -(
    pesos_perceptron[0] + pesos_perceptron[1] * x_linea
) / pesos_perceptron[2]
plt.plot(x_linea, y_linea, color="black", linewidth=2, label="Frontera aprendida")
plt.title("Perceptrón: separación de dos clases")
plt.xlabel("Característica 1")
plt.ylabel("Característica 2")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

**Cómo interpretar el resultado:** si los errores llegan a cero, el perceptrón encontró una línea capaz de separar los dos grupos. Este modelo funciona bien aquí porque los datos son linealmente separables.

## 4. Experimento 2 — Neurona sigmoide para clasificación binaria

Ahora se utiliza el conjunto Wisconsin Breast Cancer. Cada registro contiene mediciones de una muestra y pertenece a una de dos clases.

Antes de entrenar, las características se estandarizan. Esto evita que una variable domine a las demás únicamente por tener valores numéricos más grandes.

In [ ]:
datos_cancer = load_breast_cancer()

X_entreno_sig, X_prueba_sig, y_entreno_sig, y_prueba_sig = train_test_split(
    datos_cancer.data,
    datos_cancer.target,
    test_size=0.25,
    stratify=datos_cancer.target,
    random_state=SEED,
)

# El escalador aprende únicamente con el conjunto de entrenamiento.
escalador_sig = StandardScaler()
X_entreno_sig = escalador_sig.fit_transform(X_entreno_sig)
X_prueba_sig = escalador_sig.transform(X_prueba_sig)

rng = np.random.default_rng(SEED)
pesos_sig = rng.normal(0, 0.03, X_entreno_sig.shape[1])
sesgo_sig = 0.0
tasa_sig = 0.04
perdidas_sig = []

for epoca in range(900):
    prob_entreno = sigmoide(X_entreno_sig @ pesos_sig + sesgo_sig)
    error = prob_entreno - y_entreno_sig

    # Los gradientes indican cómo debe cambiar cada parámetro.
    grad_pesos = X_entreno_sig.T @ error / len(X_entreno_sig)
    grad_sesgo = error.mean()

    # Regularización pequeña para evitar pesos innecesariamente grandes.
    pesos_sig -= tasa_sig * (grad_pesos + 1e-4 * pesos_sig)
    sesgo_sig -= tasa_sig * grad_sesgo

    perdidas_sig.append(
        entropia_cruzada_binaria(y_entreno_sig, prob_entreno)
    )

prob_prueba_sig = sigmoide(X_prueba_sig @ pesos_sig + sesgo_sig)
pred_sigmoide = (prob_prueba_sig >= 0.5).astype(int)
acc_sigmoide = accuracy_score(y_prueba_sig, pred_sigmoide)

print(f"Pérdida inicial: {perdidas_sig[0]:.4f}")
print(f"Pérdida final: {perdidas_sig[-1]:.4f}")
print(f"Exactitud en prueba: {acc_sigmoide:.2%}")

plt.figure(figsize=(7, 4))
plt.plot(perdidas_sig, color="#6F2DBD")
plt.title("Neurona sigmoide: reducción de la pérdida")
plt.xlabel("Época")
plt.ylabel("Entropía cruzada")
plt.grid(alpha=0.25)
plt.show()

**Cómo interpretar el resultado:** una pérdida descendente indica que la neurona está ajustando sus pesos en la dirección correcta. La exactitud se calcula únicamente con datos que no participaron en el entrenamiento.

## 5. Experimento 3 — Red multicapa con backpropagation manual

El conjunto Two Moons no puede separarse correctamente mediante una sola línea. Por eso se implementa una red `2-16-8-1`:

- 2 valores de entrada;
- primera capa oculta de 16 neuronas ReLU;
- segunda capa oculta de 8 neuronas ReLU;
- una salida sigmoide.

El bloque de backpropagation aplica la regla de la cadena desde la salida hasta la primera capa. Así se calcula cuánto contribuyó cada peso al error.

In [ ]:
X_mlp, y_mlp = make_moons(n_samples=900, noise=0.20, random_state=SEED)

X_entreno_mlp, X_prueba_mlp, y_entreno_mlp, y_prueba_mlp = train_test_split(
    X_mlp,
    y_mlp.reshape(-1, 1),
    test_size=0.25,
    stratify=y_mlp,
    random_state=SEED,
)

escalador_mlp = StandardScaler()
X_entreno_mlp = escalador_mlp.fit_transform(X_entreno_mlp)
X_prueba_mlp = escalador_mlp.transform(X_prueba_mlp)

rng = np.random.default_rng(SEED)

# Parámetros de la primera capa: 2 entradas -> 16 neuronas.
W1 = rng.normal(0, 1.0, (2, 16))
b1 = np.zeros((1, 16))

# Parámetros de la segunda capa: 16 -> 8 neuronas.
W2 = rng.normal(0, 0.3, (16, 8))
b2 = np.zeros((1, 8))

# Parámetros de salida: 8 -> 1 probabilidad.
W3 = rng.normal(0, 0.3, (8, 1))
b3 = np.zeros((1, 1))

tasa_mlp = 0.055
perdidas_mlp = []

for epoca in range(1800):
    # -------- Propagación hacia adelante --------
    z1 = X_entreno_mlp @ W1 + b1
    a1 = np.maximum(0, z1)               # ReLU

    z2 = a1 @ W2 + b2
    a2 = np.maximum(0, z2)               # ReLU

    z3 = a2 @ W3 + b3
    prob_mlp = sigmoide(z3)              # Salida binaria

    # -------- Propagación hacia atrás --------
    dz3 = (prob_mlp - y_entreno_mlp) / len(X_entreno_mlp)
    dW3 = a2.T @ dz3
    db3 = dz3.sum(axis=0, keepdims=True)

    dz2 = (dz3 @ W3.T) * (z2 > 0)        # Derivada de ReLU
    dW2 = a1.T @ dz2
    db2 = dz2.sum(axis=0, keepdims=True)

    dz1 = (dz2 @ W2.T) * (z1 > 0)        # Derivada de ReLU
    dW1 = X_entreno_mlp.T @ dz1
    db1 = dz1.sum(axis=0, keepdims=True)

    # -------- Actualización de parámetros --------
    W3 -= tasa_mlp * dW3
    b3 -= tasa_mlp * db3
    W2 -= tasa_mlp * dW2
    b2 -= tasa_mlp * db2
    W1 -= tasa_mlp * dW1
    b1 -= tasa_mlp * db1

    if epoca % 20 == 0:
        perdidas_mlp.append(
            entropia_cruzada_binaria(y_entreno_mlp, prob_mlp)
        )


def predecir_mlp_numpy(X):
    '''Ejecuta la red entrenada sobre nuevas observaciones.'''
    capa_1 = np.maximum(0, X @ W1 + b1)
    capa_2 = np.maximum(0, capa_1 @ W2 + b2)
    return sigmoide(capa_2 @ W3 + b3)


pred_mlp = (predecir_mlp_numpy(X_prueba_mlp) >= 0.5).astype(int)
acc_mlp = accuracy_score(y_prueba_mlp, pred_mlp)

print(f"Exactitud de la MLP NumPy: {acc_mlp:.2%}")
print(f"Última pérdida registrada: {perdidas_mlp[-1]:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(np.arange(len(perdidas_mlp)) * 20, perdidas_mlp, color="#6F2DBD")
axes[0].set_title("Pérdida durante el entrenamiento")
axes[0].set_xlabel("Época")
axes[0].set_ylabel("Entropía cruzada")
axes[0].grid(alpha=0.25)

# La malla permite observar la frontera no lineal aprendida.
x_min, x_max = X_prueba_mlp[:, 0].min() - 0.5, X_prueba_mlp[:, 0].max() + 0.5
y_min, y_max = X_prueba_mlp[:, 1].min() - 0.5, X_prueba_mlp[:, 1].max() + 0.5
xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 250),
    np.linspace(y_min, y_max, 250),
)
malla = np.column_stack([xx.ravel(), yy.ravel()])
zona = (predecir_mlp_numpy(malla) >= 0.5).reshape(xx.shape)

axes[1].contourf(xx, yy, zona, alpha=0.25, cmap="coolwarm")
axes[1].scatter(
    X_prueba_mlp[:, 0], X_prueba_mlp[:, 1],
    c=y_prueba_mlp.ravel(), cmap="coolwarm", edgecolor="white", s=35
)
axes[1].set_title("Frontera no lineal aprendida")
axes[1].set_xlabel("Característica 1")
axes[1].set_ylabel("Característica 2")

plt.tight_layout()
plt.show()

**Cómo interpretar el resultado:** las capas ocultas transforman los datos hasta hacer posible una frontera curva. Esta es la ventaja principal frente al perceptrón de una sola capa.

## 6. Preparación de MNIST

MNIST contiene imágenes de 28 × 28 píxeles con dígitos escritos a mano. Cada píxel se normaliza al intervalo `[0, 1]`.

Para que la práctica pueda ejecutarse rápidamente en Colab, se utiliza una muestra reproducible de 12.000 imágenes de entrenamiento y 3.000 de prueba.

In [ ]:
(x_mnist_completo, y_mnist_completo), (x_test_completo, y_test_completo) = (
    tf.keras.datasets.mnist.load_data()
)

rng = np.random.default_rng(SEED)
indices_entreno = rng.choice(len(x_mnist_completo), 12_000, replace=False)
indices_prueba = rng.choice(len(x_test_completo), 3_000, replace=False)

x_mnist = x_mnist_completo[indices_entreno].astype("float32") / 255.0
y_mnist = y_mnist_completo[indices_entreno].astype("int64")
x_test_mnist = x_test_completo[indices_prueba].astype("float32") / 255.0
y_test_mnist = y_test_completo[indices_prueba].astype("int64")

print("Entrenamiento:", x_mnist.shape, y_mnist.shape)
print("Prueba:", x_test_mnist.shape, y_test_mnist.shape)

# Muestra visual equivalente a la presentada en el informe.
fig, axes = plt.subplots(4, 5, figsize=(11, 8))
for eje, imagen, etiqueta in zip(axes.ravel(), x_mnist[:20], y_mnist[:20]):
    eje.imshow(imagen, cmap="gray")
    eje.set_title(f"Número: {etiqueta}")
    eje.axis("off")

plt.suptitle("Muestra de imágenes MNIST", fontsize=16)
plt.tight_layout()
plt.show()

## 7. TensorFlow + Keras

La imagen de 28 × 28 se aplana para formar un vector de 784 valores. Después pasa por dos capas ocultas de 160 y 80 neuronas. La capa final entrega diez **logits**, uno por cada posible dígito.

`Dropout(0.15)` desactiva aleatoriamente algunas conexiones durante el entrenamiento para reducir el sobreajuste.

In [ ]:
modelo_keras = tf.keras.Sequential(
    [
        tf.keras.layers.Input(shape=(28, 28)),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(CAPAS_OCULTAS[0], activation="relu"),
        tf.keras.layers.Dropout(0.15),
        tf.keras.layers.Dense(CAPAS_OCULTAS[1], activation="relu"),
        tf.keras.layers.Dense(10),  # Diez salidas: una por dígito.
    ],
    name="mlp_densa_mnist",
)

modelo_keras.compile(
    optimizer="adam",
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)

modelo_keras.summary()

historial_keras = modelo_keras.fit(
    x_mnist,
    y_mnist,
    validation_split=0.15,
    epochs=3,
    batch_size=128,
    verbose=2,
)

perdida_keras, acc_keras = modelo_keras.evaluate(
    x_test_mnist, y_test_mnist, verbose=0
)

logits_keras = modelo_keras.predict(x_test_mnist, verbose=0)
pred_keras = logits_keras.argmax(axis=1)

print(f"Pérdida de prueba Keras: {perdida_keras:.4f}")
print(f"Exactitud de prueba Keras: {acc_keras:.2%}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(historial_keras.history["loss"], marker="o", label="Entrenamiento")
axes[0].plot(historial_keras.history["val_loss"], marker="o", label="Validación")
axes[0].set_title("Keras: evolución de la pérdida")
axes[0].set_xlabel("Época")
axes[0].set_ylabel("Pérdida")
axes[0].legend()
axes[0].grid(alpha=0.25)

axes[1].plot(historial_keras.history["accuracy"], marker="o", label="Entrenamiento")
axes[1].plot(historial_keras.history["val_accuracy"], marker="o", label="Validación")
axes[1].set_title("Keras: evolución de la exactitud")
axes[1].set_xlabel("Época")
axes[1].set_ylabel("Exactitud")
axes[1].legend()
axes[1].grid(alpha=0.25)

plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 8))
ConfusionMatrixDisplay.from_predictions(
    y_test_mnist,
    pred_keras,
    cmap="Purples",
    colorbar=False,
    ax=ax,
)
ax.set_title("Matriz de confusión — TensorFlow + Keras")
plt.show()

## 8. PyTorch

Se construye la misma arquitectura `784-160-80-10`. La diferencia principal es que PyTorch hace explícitos el ciclo de entrenamiento, el cálculo de la pérdida y la actualización de los parámetros.

La instrucción `loss.backward()` ejecuta automáticamente la propagación hacia atrás.

In [ ]:
class RedDensaPyTorch(nn.Module):
    '''Arquitectura densa equivalente a la utilizada en Keras.'''

    def __init__(self):
        super().__init__()
        self.capas = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, CAPAS_OCULTAS[0]),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(CAPAS_OCULTAS[0], CAPAS_OCULTAS[1]),
            nn.ReLU(),
            nn.Linear(CAPAS_OCULTAS[1], 10),
        )

    def forward(self, x):
        return self.capas(x)


modelo_torch = RedDensaPyTorch()
optimizador_torch = torch.optim.Adam(modelo_torch.parameters(), lr=1e-3)
funcion_perdida_torch = nn.CrossEntropyLoss()

dataset_torch = TensorDataset(
    torch.from_numpy(x_mnist),
    torch.from_numpy(y_mnist),
)
cargador_torch = DataLoader(
    dataset_torch,
    batch_size=128,
    shuffle=True,
)

perdidas_torch = []
exactitudes_torch = []

for epoca in range(3):
    modelo_torch.train()
    perdida_acumulada = 0.0
    aciertos = 0
    observaciones = 0

    for imagenes_lote, etiquetas_lote in cargador_torch:
        optimizador_torch.zero_grad()

        logits = modelo_torch(imagenes_lote)
        perdida = funcion_perdida_torch(logits, etiquetas_lote)

        # Backpropagation automático y actualización de pesos.
        perdida.backward()
        optimizador_torch.step()

        perdida_acumulada += perdida.item() * len(imagenes_lote)
        aciertos += (logits.argmax(dim=1) == etiquetas_lote).sum().item()
        observaciones += len(imagenes_lote)

    perdida_epoca = perdida_acumulada / observaciones
    exactitud_epoca = aciertos / observaciones
    perdidas_torch.append(perdida_epoca)
    exactitudes_torch.append(exactitud_epoca)

    print(
        f"Época {epoca + 1}: "
        f"pérdida={perdida_epoca:.4f}, "
        f"exactitud={exactitud_epoca:.2%}"
    )

In [ ]:
modelo_torch.eval()

with torch.no_grad():
    logits_prueba_torch = modelo_torch(torch.from_numpy(x_test_mnist))
    perdida_prueba_torch = funcion_perdida_torch(
        logits_prueba_torch, torch.from_numpy(y_test_mnist)
    ).item()
    pred_torch = logits_prueba_torch.argmax(dim=1).numpy()

acc_torch = accuracy_score(y_test_mnist, pred_torch)

print(f"Pérdida de prueba PyTorch: {perdida_prueba_torch:.4f}")
print(f"Exactitud de prueba PyTorch: {acc_torch:.2%}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(range(1, 4), perdidas_torch, marker="o", color="#2C8C6D")
axes[0].set_title("PyTorch: pérdida de entrenamiento")
axes[0].set_xlabel("Época")
axes[0].set_ylabel("Pérdida")
axes[0].grid(alpha=0.25)

axes[1].plot(range(1, 4), exactitudes_torch, marker="o", color="#6F2DBD")
axes[1].set_title("PyTorch: exactitud de entrenamiento")
axes[1].set_xlabel("Época")
axes[1].set_ylabel("Exactitud")
axes[1].grid(alpha=0.25)

plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 8))
ConfusionMatrixDisplay.from_predictions(
    y_test_mnist,
    pred_torch,
    cmap="Greens",
    colorbar=False,
    ax=ax,
)
ax.set_title("Matriz de confusión — PyTorch")
plt.show()

## 9. Comparación final

Esta celda reúne las exactitudes calculadas durante la ejecución. No son valores escritos manualmente: provienen de los modelos entrenados en las celdas anteriores.

In [ ]:
nombres_modelos = [
    "Perceptrón",
    "Neurona sigmoide",
    "MLP NumPy",
    "Keras MNIST",
    "PyTorch MNIST",
]

exactitudes = [
    acc_perceptron,
    acc_sigmoide,
    acc_mlp,
    acc_keras,
    acc_torch,
]

print("Resumen de resultados")
print("-" * 48)
for nombre, exactitud in zip(nombres_modelos, exactitudes):
    print(f"{nombre:<22} {exactitud:>8.2%}")

plt.figure(figsize=(9, 4.8))
colores = ["#6F2DBD", "#8F62C9", "#B091DB", "#2C8C6D", "#5EAE91"]
barras = plt.barh(nombres_modelos, exactitudes, color=colores)
plt.xlim(0.80, 1.01)
plt.xlabel("Exactitud")
plt.title("Comparación de los cinco modelos")
plt.grid(axis="x", alpha=0.2)

for barra, valor in zip(barras, exactitudes):
    plt.text(valor - 0.005, barra.get_y() + barra.get_height() / 2,
             f"{valor:.2%}", ha="right", va="center", color="white", weight="bold")

plt.tight_layout()
plt.show()

## 10. Conclusiones

- El perceptrón funciona cuando las clases pueden separarse mediante una línea.
- La salida sigmoide permite expresar una probabilidad y entrenar con entropía cruzada.
- Las capas ocultas y ReLU permiten aprender relaciones no lineales.
- Backpropagation convierte el error final en correcciones para cada matriz de pesos.
- Keras y PyTorch producen resultados comparables al utilizar los mismos datos y una arquitectura equivalente.
- Los errores de MNIST se concentran principalmente en dígitos con formas visualmente parecidas.

## Referencias

- Abadi, M., Barham, P., Chen, J., et al. (2016). TensorFlow: A system for large-scale machine learning. *12th USENIX Symposium on Operating Systems Design and Implementation*, 265–283.
- Goodfellow, I., Bengio, Y., & Courville, A. (2016). *Deep learning*. MIT Press. https://www.deeplearningbook.org/
- LeCun, Y., Cortes, C., & Burges, C. (s. f.). *The MNIST database of handwritten digits*. http://yann.lecun.com/exdb/mnist/
- PyTorch. (2026). *PyTorch tutorials*. https://docs.pytorch.org/tutorials/
- TensorFlow. (2026). *TensorFlow 2 quickstart for beginners*. https://www.tensorflow.org/tutorials/quickstart/beginner